# Repeated-structure identification stress tests

The frozen estimator assumes unordered partial observations of repeated physical structures share an ordered canonical coordinate. Test its implementation and identification ceiling against **known simulated position**. This is an optimistic synthetic challenge, not anatomical validation of the PT data. True fine position enters evaluation only; exact coarse labels enter training. No depth information, query labels or query range registration enters inference.

The experiment was specified in the research ledger before outputs: three seeds, spatial-only/independent/coupled nuisance references, full and partial query coverage, reversed nuisance coupling, and paired quarter-read thinning. Preserve failures and compare frozen PC1 and soft coarse-segment prediction.

In [ ]:
from pathlib import Path
import hashlib,io,json,os,sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_structure_simulation import simulate_repeated_sections
from pseudospace.pseudostructure_atlas import fit_pseudostructure_atlas,project_pseudostructure_atlas
from pseudospace.count_representation import transform_count_representation
from pseudospace.repeated_atlas_baselines import fit_pc1_reference,project_pc1_reference,fit_segment_mixture,predict_segment_mixture
from pseudospace.stage_cache import cached_payload
NOTEBOOK_LOGIC_VERSION='19.identification_stress.2'
SEEDS=(19,41,73)
OUT=Path(os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results'))).expanduser()/'repeated_structure_identification_stress'
OUT.mkdir(parents=True,exist_ok=True)
FIT_PARAMS=dict(n_components=15,seed=15,grid=np.linspace(0,1,101),roughness=.001,
    offset_penalty=4.,anatomy_strength=1.,anatomy_width=.1,max_iter=60,tol=1e-5,
    representation_theta=100.,count_theta=100.,rate_bandwidth=.08,min_effective=12.)
CONFIG=dict(version=NOTEBOOK_LOGIC_VERSION,seeds=SEEDS,n_train_per_specimen=500,
    n_query=400,spatial_strength=3.,nuisance_strength=3.,realization_sd=.1,
    partial_interval=[.35,.65],thin_fraction=.25,
    fit={**FIT_PARAMS,'grid':FIT_PARAMS['grid'].tolist()})
MODULES=['repeated_structure_simulation.py','pseudostructure_atlas.py','count_representation.py',
    'joint_repeated_atlas.py','count_rate_atlas.py','repeated_atlas_baselines.py']
CODE_DIGEST=hashlib.sha256(b''.join((REPO/'pseudospace'/f).read_bytes() for f in MODULES)).hexdigest()
(OUT/'manifest.json').write_text(json.dumps(dict(config=CONFIG,code_digest=CODE_DIGEST),indent=2))

## Independent section generation and frozen predictors

Forty genes carry orthogonal spatial and nuisance logit directions. Counts are multinomial at variable exposure; the inference likelihood remains its existing Gaussian/NB approximation. Independent section-level gene noise stands in for integrated realization variation. The generator does not simulate longitudinal dependence within identifiable nephrons.

The coupled nuisance program can teach an apparently reliable ordering in training. Reversing that coupling at query time tests a domain change that violates the shared molecular path. Within-segment ranking is reported where at least eight query sections exist. Absolute error uses the simulator's numerical gauge and is secondary to ranking; the model does not estimate physical length.

In [ ]:
def predictors(reference,query,pc,mixture):
    projection=project_pseudostructure_atlas(reference,query['counts'],query['library'],
        query['gene_names'],query['structure_ids'])
    Y=transform_count_representation(query['counts'],query['library'],reference['representation'])
    coordinates={name:projection[name]['z_mean'] for name in ('gaussian','count')}
    coordinates['count_PC1']=project_pc1_reference(pc,Y)
    coordinates['soft_segment']=predict_segment_mixture(mixture,Y)['prediction'][:,0]
    return projection,coordinates

def evaluation_rows(query,projection,coordinates,key,original=None):
    rows=[]
    for method,z in coordinates.items():
        for segment in [None,0,1,2]:
            use=np.ones(len(z),bool) if segment is None else query['anatomy']==segment
            if use.sum()<8: continue
            truth=query['true_z'][use]
            varying=np.ptp(z[use])>0 and np.ptp(truth)>0
            row=dict(**key,method=method,segment='all' if segment is None else ['S1','S2','S3'][segment],
                n=int(use.sum()),spearman=float(spearmanr(z[use],truth).statistic) if varying else np.nan,
                gauge_mae=float(np.mean(np.abs(z[use]-truth))),
                z_q05=float(np.quantile(z[use],.05)),z_q95=float(np.quantile(z[use],.95)),
                endpoint_fraction=float(np.mean((z[use]<.01)|(z[use]>.99))))
            if method in ('gaussian','count'):
                row['conditional_entropy']=float(projection[method]['entropy'][use].mean())
                row['model_disagreement']=float(projection['coordinate_disagreement'][use].mean())
            if original is not None:
                row['paired_thinning_mae']=float(np.mean(np.abs(z[use]-original[method][use])))
            rows.append(row)
    return rows

def subset_query(query,use):
    return {key:value[use] if key!='gene_names' else value.copy() for key,value in query.items()}

def run_stress_tests():
    scores,checks,failures=[],[],[]
    for seed in SEEDS:
        for scenario,mode,strength in [('spatial_only','independent',0.),
                ('independent_nuisance','independent',3.),('coupled_nuisance','coupled',3.)]:
            key=dict(seed=seed,training_scenario=scenario)
            try:
                train=simulate_repeated_sections(seed=seed,n_per_specimen=500,
                    specimens=('train_A','train_B'),nuisance_mode=mode,nuisance_strength=strength)
                reference=fit_pseudostructure_atlas(train['counts'],train['library'],train['specimens'],
                    train['anatomy'],train['gene_names'],np.ones(40,bool),**FIT_PARAMS)
                Y=transform_count_representation(train['counts'],train['library'],reference['representation'])
                pc=fit_pc1_reference(Y,train['anatomy'])
                mixture=fit_segment_mixture(Y,train['anatomy'][:,None]/2.,train['anatomy'],
                    train['specimens'],C=1.,seed=15)
                query_modes=[mode,'reversed'] if mode=='coupled' else [mode]
                for query_mode in query_modes:
                    for coverage,interval in [('full',(0.,1.)),('partial',(.35,.65))]:
                        query=simulate_repeated_sections(seed=seed+1000,n_per_specimen=400,
                            specimens=('unseen',),interval=interval,nuisance_mode=query_mode,nuisance_strength=strength)
                        projection,coordinates=predictors(reference,query,pc,mixture)
                        meta=dict(**key,query_mode=query_mode,coverage=coverage,measurement='original')
                        scores.extend(evaluation_rows(query,projection,coordinates,meta))
                        # Changing population composition cannot change the same profiles.
                        use=(query['true_z']>=.35)&(query['true_z']<=.65)
                        sub_projection,sub_coordinates=predictors(reference,subset_query(query,use),pc,mixture)
                        maximum=max(float(np.max(np.abs(sub_coordinates[m]-coordinates[m][use]))) for m in coordinates)
                        checks.append(dict(**key,query_mode=query_mode,coverage=coverage,
                            subset_max_change=maximum,converged=bool(reference['gaussian']['converged'])))
                        rng=np.random.default_rng(seed+2000)
                        thinned={k:v.copy() for k,v in query.items()}
                        thinned['counts']=rng.binomial(query['counts'],.25)
                        thinned['library']=thinned['counts'].sum(axis=1)
                        if np.any(thinned['library']==0): raise ValueError('Empty thinned library')
                        thin_projection,thin_coordinates=predictors(reference,thinned,pc,mixture)
                        scores.extend(evaluation_rows(thinned,thin_projection,thin_coordinates,
                            {**meta,'measurement':'quarter_reads'},original=coordinates))
                print('Completed synthetic reference:',key,flush=True)
            except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
                failures.append(dict(**key,error=str(exc)))
                print('Synthetic reference failure:',failures[-1],flush=True)
    return {k:pd.DataFrame(v).to_json(orient='table')
        for k,v in dict(scores=scores,checks=checks,failures=failures).items()}

In [ ]:
payload=cached_payload('synthetic_repeated_structure_identification',run_stress_tests,
    root=OUT/'stage_cache',params=CONFIG,inputs={'simulation_seeds':np.array(SEEDS)},code=CODE_DIGEST)
results={key:pd.read_json(io.StringIO(str(value)),orient='table') for key,value in payload.items()}
for key,frame in results.items(): frame.to_csv(OUT/(key+'.csv'),index=False)
display(results['failures'])
if not results['checks'].empty:
    if results['checks'].subset_max_change.max()>1e-10:
        raise RuntimeError('Frozen projection depends on query population composition')
    if not results['checks'].converged.all(): raise RuntimeError('Nonconverged reference reported as successful')
# Failed fits stay visible; they are a result, rather than silently rerun with tuned parameters.
print('Failures:',len(results['failures']),'of',len(SEEDS)*3,'prespecified references')

## Read identification and failure together

Compare methods and scenarios without selecting parameters from these outcomes. A successful full-axis rank is insufficient if fine within-segment rank, partial coverage or nuisance shifts fail. Neither conditional entropy nor reproducibility alone proves physical spatial identification. Full and restricted populations are different simulated draws at different true locations; only the same-profile subset check is an exact population-invariance requirement.

In [ ]:
scores=results['scores']
if not scores.empty:
    summary=scores.query("segment == 'all' and measurement == 'original'").groupby(
        ['training_scenario','query_mode','coverage','method'],sort=False)[
        ['spearman','gauge_mae','z_q05','z_q95','endpoint_fraction','conditional_entropy','model_disagreement']].mean()
    display(summary)
    display(scores.query("segment != 'all' and measurement == 'original'").pivot_table(
        index=['training_scenario','query_mode','coverage','segment'],columns='method',values='spearman'))
    display(scores.query("segment == 'all' and measurement == 'quarter_reads'").pivot_table(
        index=['training_scenario','query_mode','coverage'],columns='method',values='paired_thinning_mae'))
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
    for ax,coverage in zip(axes,['full','partial']):
        panel=scores.query("segment == 'all' and measurement == 'original' and coverage == @coverage").copy()
        panel['condition']=panel.training_scenario+' / '+panel.query_mode
        names=list(dict.fromkeys(panel.condition))
        for j,method in enumerate(['gaussian','count','count_PC1','soft_segment']):
            values=[panel.loc[(panel.condition==name)&(panel.method==method),'spearman'].mean() for name in names]
            ax.plot(np.arange(len(names)),values,marker='o',label=method)
        ax.set_xticks(np.arange(len(names)),names,rotation=35,ha='right',fontsize=8)
        ax.set_title(coverage+' query coverage');ax.axhline(0,color='grey',linewidth=.7)
        ax.set_ylim(-1.05,1.05)
    axes[0].set_ylabel('True-position Spearman rank (mean of 3 seeds)')
    axes[1].legend(fontsize=8)
    fig.suptitle('Independent-nuisance atlas fits failed in all three seeds; omitted from this plot',fontsize=9)
    fig.tight_layout();plt.show()

## Diagnose independent-nuisance failures without accepting their coordinates

All three independent-nuisance references failed the public convergence guard. Repeat exactly those settings through the underlying solver to expose objective history and representation alignment. Score the frozen PC1 and soft segment baselines on the same independent query data even though the atlas failed. Nonconverged atlas coordinates do not become accepted predictions, and this diagnostic does not change iteration limits or tolerance.

In [ ]:
from pseudospace.count_representation import fit_count_representation
from pseudospace.joint_repeated_atlas import fit_joint_atlas
FAILED_DIAGNOSTIC_VERSION='19.independent_failure_diagnostic.1'
def diagnose_independent_nuisance():
    diagnostic_rows,history_rows,baseline_rows=[],[],[]
    for seed in SEEDS:
        train=simulate_repeated_sections(seed=seed,n_per_specimen=500,
            specimens=('train_A','train_B'),nuisance_strength=3.,nuisance_mode='independent')
        Y,transform=fit_count_representation(train['counts'],train['library'],train['specimens'],
            n_components=15,theta=100.,seed=15)
        pc=fit_pc1_reference(Y,train['anatomy'])
        initial=project_pc1_reference(pc,Y)
        mixture=fit_segment_mixture(Y,train['anatomy'][:,None]/2.,train['anatomy'],train['specimens'],C=1.,seed=15)
        solver=fit_joint_atlas(Y,train['specimens'],train['anatomy'],initial_z=initial,
            grid=FIT_PARAMS['grid'],roughness=.001,offset_penalty=4.,anatomy_strength=1.,
            anatomy_width=.1,max_iter=60,tol=1e-5)
        diagnostic_rows.append(dict(seed=seed,converged=solver['converged'],iterations=len(solver['history']),
            PC1_position_rank=float(spearmanr(initial,train['true_z']).statistic),
            PC1_nuisance_abs_rank=float(abs(spearmanr(initial,train['nuisance_q']).statistic)),
            final_objective_change=solver['history'][-1]['objective_change']))
        history_rows.extend(dict(seed=seed,**row) for row in solver['history'])
        for coverage,interval in [('full',(0.,1.)),('partial',(.35,.65))]:
            query=simulate_repeated_sections(seed=seed+1000,n_per_specimen=400,specimens=('unseen',),
                interval=interval,nuisance_strength=3.,nuisance_mode='independent')
            query_Y=transform_count_representation(query['counts'],query['library'],transform)
            coordinates=dict(count_PC1=project_pc1_reference(pc,query_Y),
                soft_segment=predict_segment_mixture(mixture,query_Y)['prediction'][:,0])
            baseline_rows.extend(evaluation_rows(query,{},coordinates,
                dict(seed=seed,training_scenario='independent_nuisance',query_mode='independent',
                    coverage=coverage,measurement='original')))
    return {key:pd.DataFrame(value).to_json(orient='table') for key,value in
        dict(diagnostics=diagnostic_rows,history=history_rows,baselines=baseline_rows).items()}
failure_payload=cached_payload('independent_nuisance_failure_diagnostic',diagnose_independent_nuisance,
    root=OUT/'stage_cache',params=dict(config=CONFIG,diagnostic_version=FAILED_DIAGNOSTIC_VERSION),
    inputs={'seeds':np.array(SEEDS)},code=CODE_DIGEST)
failure_diagnostics={key:pd.read_json(io.StringIO(str(value)),orient='table') for key,value in failure_payload.items()}
for key,frame in failure_diagnostics.items():frame.to_csv(OUT/('independent_failure_'+key+'.csv'),index=False)
display(failure_diagnostics['diagnostics'])
display(failure_diagnostics['baselines'].pivot_table(index=['coverage','segment'],columns='method',values='spearman'))
print('These diagnostics retain the original convergence requirement; no failed atlas is accepted.')